# Day 05 — Dates and Time Series
**Estimated time:** 60–75 min  
**Datasets:** `sales_orders.csv`, `materials_inventory.csv`

## Learning Objectives
- Parse, manipulate, and decompose datetime columns in pandas
- Build time-series aggregations with `resample()` and rolling windows
- Perform date arithmetic for aging, recency, and trend analysis


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../datasets"

sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce")

inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LAST_MOVEMENT_DATE"] = pd.to_datetime(inv["LAST_MOVEMENT_DATE"], errors="coerce")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce")

print("Sales date range:", sales["ERDAT"].min(), "→", sales["ERDAT"].max())
print("Null dates in sales:", sales["ERDAT"].isna().sum())


In [ ]:
# ── 1. Datetime decomposition ────────────────────────────────────────────────
sales["year"]    = sales["ERDAT"].dt.year
sales["month"]   = sales["ERDAT"].dt.month
sales["quarter"] = sales["ERDAT"].dt.quarter
sales["weekday"] = sales["ERDAT"].dt.day_name()
sales["ym"]      = sales["ERDAT"].dt.to_period("M")   # Year-Month period

print(sales[["ERDAT","year","month","quarter","weekday","ym"]].head(8))


In [ ]:
# ── 2. resample() — monthly revenue aggregation ──────────────────────────────
# Must set datetime index for resample
ts = (sales.set_index("ERDAT")
           .resample("ME")["NETWR"]   # ME = Month End
           .agg(["sum","count","mean"])
           .rename(columns={"sum":"revenue","count":"order_count","mean":"avg_order_value"})
           .dropna())

print(f"Monthly buckets: {len(ts)}")
display(ts.tail(18))


In [ ]:
# ── 3. Quarterly aggregation ──────────────────────────────────────────────────
quarterly = (sales.set_index("ERDAT")
                  .resample("QE")["NETWR"]
                  .sum()
                  .rename("quarterly_revenue")
                  .to_frame())
display(quarterly)


In [ ]:
# ── 4. Rolling 30-day revenue ────────────────────────────────────────────────
# Daily revenue first, then rolling
daily_rev = (sales.set_index("ERDAT")["NETWR"]
                  .resample("D")
                  .sum()
                  .fillna(0))

rolling_30 = daily_rev.rolling(window=30, min_periods=1).mean().rename("rolling_30d_avg")

fig, ax = plt.subplots(figsize=(12, 4))
daily_rev.plot(ax=ax, alpha=0.3, label="Daily Revenue", color="steelblue")
rolling_30.plot(ax=ax, label="30-Day Rolling Avg", color="firebrick", linewidth=2)
ax.set_title("Daily Revenue + 30-Day Rolling Average")
ax.set_ylabel("Revenue")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# ── 5. Date arithmetic: aging analysis on inventory ──────────────────────────
today = pd.Timestamp.today().normalize()

inv["days_since_movement"] = (today - inv["LAST_MOVEMENT_DATE"]).dt.days

def aging_bucket(days):
    if pd.isna(days):  return "Unknown"
    if days < 30:      return "< 30 days"
    if days < 90:      return "30–90 days"
    if days < 180:     return "90–180 days"
    return "> 180 days"

inv["aging_bucket"] = inv["days_since_movement"].apply(aging_bucket)
aging_order = ["< 30 days", "30–90 days", "90–180 days", "> 180 days", "Unknown"]
inv["aging_bucket"] = pd.Categorical(inv["aging_bucket"], categories=aging_order, ordered=True)

print(inv.groupby("aging_bucket", observed=False)["MATNR"].count())


In [ ]:
# ── 6. Month-over-Month change ────────────────────────────────────────────────
monthly_rev = (sales.set_index("ERDAT")
                    .resample("ME")["NETWR"]
                    .sum()
                    .to_frame("revenue"))

monthly_rev["prev_revenue"] = monthly_rev["revenue"].shift(1)
monthly_rev["mom_abs"]      = monthly_rev["revenue"] - monthly_rev["prev_revenue"]
monthly_rev["mom_pct"]      = (monthly_rev["mom_abs"] / monthly_rev["prev_revenue"] * 100).round(2)

display(monthly_rev.tail(18))


In [ ]:
# ── 7. Top month and sharpest MoM decline ────────────────────────────────────
last_18 = monthly_rev.dropna().tail(18)

top_month = last_18["revenue"].idxmax()
worst_mom = last_18["mom_pct"].idxmin()

print(f"Top revenue month:       {top_month.strftime('%Y-%m')}  (${last_18.loc[top_month,'revenue']:,.0f})")
print(f"Sharpest MoM decline:    {worst_mom.strftime('%Y-%m')}  ({last_18.loc[worst_mom,'mom_pct']:.1f}%)")


---
## Daily Prompt

**Build a monthly revenue trend for the last 18 months. Identify:**
1. The top revenue month
2. The month with the sharpest MoM % decline  
3. Any months with >20% MoM growth (spike detection)

Plot all three on a single annotated line chart.

```python
# YOUR SOLUTION
last_18 = monthly_rev.dropna().tail(18)

fig, ax = plt.subplots(figsize=(13, 5))
last_18["revenue"].plot(ax=ax, marker="o", linewidth=2, color="steelblue")

# YOUR SOLUTION: annotate top_month and worst_mom on the chart
# Hint: ax.annotate("Peak", xy=(top_month, last_18.loc[top_month,"revenue"]), ...)

ax.set_title("Monthly Revenue — Last 18 Months")
ax.set_xlabel("")
ax.set_ylabel("Revenue")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
```

> **Hint:** Use `ax.annotate()` with `xytext` offset and `arrowprops` for professional-looking annotations.  
> For spike detection: `last_18[last_18["mom_pct"] > 20]`
